In [16]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import time
import sys
sys.path.append('..')

from environments.rock_paper_scissors import RockPaperScissorsEnv
from algorithms.monte_carlo import monte_carlo_es, on_policy_first_visit_mc_control, off_policy_mc_control
from algorithms.temporal_difference_learning import sarsa, q_learning
from algorithms.planning import dyna_q, dyna_q_plus

print("Imports OK ✅")

Imports OK ✅


In [17]:
ACTION_NAMES = {0: 'Pierre', 1: 'Feuille', 2: 'Ciseaux'}

algos = {
    'Monte Carlo ES': lambda: monte_carlo_es(RockPaperScissorsEnv(), num_episodes=100000),
    'On-policy MC':   lambda: on_policy_first_visit_mc_control(RockPaperScissorsEnv(), num_episodes=100000),
    'Off-policy MC':  lambda: off_policy_mc_control(RockPaperScissorsEnv(), num_episodes=100000),
    'Sarsa':          lambda: sarsa(RockPaperScissorsEnv(), num_episodes=100000),
    'Q-Learning':     lambda: q_learning(RockPaperScissorsEnv(), num_episodes=100000),
    'Dyna-Q':         lambda: dyna_q(RockPaperScissorsEnv(), max_steps=100000),
    'Dyna-Q+':        lambda: dyna_q_plus(RockPaperScissorsEnv(), max_steps=100000),
}

print("=== Tous les algos sur RockPaperScissors ===\n")

results = {}
for name, algo in algos.items():
    t = time.time()
    pi, Q = algo()
    t = time.time() - t

    r2_1 = np.argmax(Q[1])  # R1=Pierre → optimal=Feuille(1)
    r2_2 = np.argmax(Q[2])  # R1=Feuille → optimal=Ciseaux(2)
    r2_3 = np.argmax(Q[3])  # R1=Ciseaux → optimal=Pierre(0)

    print(f"{name} | temps={t:.3f}s")
    print(f"  R2 (R1=Pierre)  → {ACTION_NAMES[r2_1]}")
    print(f"  R2 (R1=Feuille) → {ACTION_NAMES[r2_2]}")
    print(f"  R2 (R1=Ciseaux) → {ACTION_NAMES[r2_3]}")
    print()

    results[name] = (pi, Q, t)

=== Tous les algos sur RockPaperScissors ===

Monte Carlo ES | temps=2.930s
  R2 (R1=Pierre)  → Feuille
  R2 (R1=Feuille) → Ciseaux
  R2 (R1=Ciseaux) → Pierre

On-policy MC | temps=1.468s
  R2 (R1=Pierre)  → Feuille
  R2 (R1=Feuille) → Ciseaux
  R2 (R1=Ciseaux) → Pierre

Off-policy MC | temps=3.058s
  R2 (R1=Pierre)  → Feuille
  R2 (R1=Feuille) → Ciseaux
  R2 (R1=Ciseaux) → Pierre

Sarsa | temps=1.300s
  R2 (R1=Pierre)  → Feuille
  R2 (R1=Feuille) → Ciseaux
  R2 (R1=Ciseaux) → Pierre

Q-Learning | temps=1.460s
  R2 (R1=Pierre)  → Feuille
  R2 (R1=Feuille) → Ciseaux
  R2 (R1=Ciseaux) → Pierre

Dyna-Q | temps=9.760s
  R2 (R1=Pierre)  → Feuille
  R2 (R1=Feuille) → Ciseaux
  R2 (R1=Ciseaux) → Pierre

Dyna-Q+ | temps=11.917s
  R2 (R1=Pierre)  → Pierre
  R2 (R1=Feuille) → Pierre
  R2 (R1=Ciseaux) → Pierre



In [18]:
print("=== Stratégie optimale attendue ===")
print("R2 (R1=Pierre)  → Feuille")
print("R2 (R1=Feuille) → Ciseaux")
print("R2 (R1=Ciseaux) → Pierre")
print()

print("=== Résultat par algo ===")
for name, (pi, Q, t) in results.items():
    r2_1 = np.argmax(Q[1])
    r2_2 = np.argmax(Q[2])
    r2_3 = np.argmax(Q[3])
    optimal = (r2_1 == 1 and r2_2 == 2 and r2_3 == 0)
    print(f"{name} → optimal ? {'✅' if optimal else '❌'}")

=== Stratégie optimale attendue ===
R2 (R1=Pierre)  → Feuille
R2 (R1=Feuille) → Ciseaux
R2 (R1=Ciseaux) → Pierre

=== Résultat par algo ===
Monte Carlo ES → optimal ? ✅
On-policy MC → optimal ? ✅
Off-policy MC → optimal ? ✅
Sarsa → optimal ? ✅
Q-Learning → optimal ? ✅
Dyna-Q → optimal ? ✅
Dyna-Q+ → optimal ? ❌


In [19]:
names = list(results.keys())
times = [results[n][2] for n in names]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(names, times, color=['blue', 'green', 'orange', 'red', 'purple', 'brown', 'pink'])
ax.set_ylabel('Temps (s)')
ax.set_title('Comparaison temps exécution — RockPaperScissors')
plt.xticks(rotation=15)
for bar, t in zip(bars, times):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
            f'{t:.3f}s', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.savefig('rps_temps.png', dpi=100, bbox_inches='tight')
plt.close()
print("Graphique sauvegardé ✅")

Graphique sauvegardé ✅


In [20]:
episodes_list = [100, 500, 1000, 5000, 10000, 50000, 100000]
optimal_scores = []

for n_ep in episodes_list:
    env = RockPaperScissorsEnv()
    pi, Q = monte_carlo_es(env, num_episodes=n_ep)
    r2_1 = np.argmax(Q[1])
    r2_2 = np.argmax(Q[2])
    r2_3 = np.argmax(Q[3])
    score = sum([r2_1 == 1, r2_2 == 2, r2_3 == 0])
    optimal_scores.append(score)
    print(f"num_episodes={n_ep:7d} → états R2 optimaux : {score}/3")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(episodes_list, optimal_scores, marker='o', color='blue')
ax.set_xlabel("Nombre d'épisodes")
ax.set_ylabel('États R2 optimaux (max=3)')
ax.set_title('Impact num_episodes — Monte Carlo ES — RPS')
ax.set_xscale('log')
ax.set_yticks([0, 1, 2, 3])
plt.tight_layout()
plt.savefig('rps_mc_episodes.png', dpi=100, bbox_inches='tight')
plt.close()
print("Graphique sauvegardé ✅")

num_episodes=    100 → états R2 optimaux : 3/3
num_episodes=    500 → états R2 optimaux : 3/3
num_episodes=   1000 → états R2 optimaux : 3/3
num_episodes=   5000 → états R2 optimaux : 3/3
num_episodes=  10000 → états R2 optimaux : 3/3
num_episodes=  50000 → états R2 optimaux : 3/3
num_episodes= 100000 → états R2 optimaux : 3/3
Graphique sauvegardé ✅


In [22]:
epsilons = [0.01, 0.05, 0.1, 0.2, 0.5]
sarsa_scores = []
qlearning_scores = []

for eps in epsilons:
    pi, Q = sarsa(RockPaperScissorsEnv(), epsilon=eps, num_episodes=100000)
    score = sum([np.argmax(Q[1])==1, np.argmax(Q[2])==2, np.argmax(Q[3])==0])
    sarsa_scores.append(score)

    pi, Q = q_learning(RockPaperScissorsEnv(), epsilon=eps, num_episodes=100000)
    score = sum([np.argmax(Q[1])==1, np.argmax(Q[2])==2, np.argmax(Q[3])==0])
    qlearning_scores.append(score)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(epsilons, sarsa_scores, marker='o', label='Sarsa', color='blue')
ax.plot(epsilons, qlearning_scores, marker='o', label='Q-Learning', color='orange')
ax.set_xlabel('Epsilon')
ax.set_ylabel('États R2 optimaux (max=3)')
ax.set_title('Impact de epsilon — Sarsa vs Q-Learning — RPS')
ax.legend()
ax.set_yticks([0, 1, 2, 3])
plt.tight_layout()
plt.savefig('rps_epsilon.png', dpi=100, bbox_inches='tight')
plt.close()
print("Graphique sauvegardé ✅")

Graphique sauvegardé ✅
